# CVE Feature Engineering & Weak Label Construction

**Purpose**: Transform raw CVE data into ML-ready features and construct weak supervision labels

**What this notebook does**:
1. **Feature Engineering** - Create 12-14 features from CVE enrichments
2. **Weak Label Construction** - Build graded labels (0-5) with confidence scores
3. **Label Diagnostics** - Analyze label quality and distribution
4. **Feature Analysis** - Missingness, correlations, importance preview
5. **Data Preparation** - Save processed features for model training

**Key Innovation**: Confidence-weighted weak supervision
- Each label has a confidence score (0-1) based on signal reliability
- KEV flags have high confidence (1.0) - known exploited vulnerabilities
- Heuristic signals have lower confidence (0.3-0.7)
- Model training uses confidence to weight importance

---

## 1. Setup & Imports

In [27]:
import sys
import os
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

warnings.filterwarnings('ignore')

# Setup project paths
project_root = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(project_root))
os.chdir(project_root)

# Import project modules
from src.core.cve_database import CVEDatabase
from src.features.engineering import create_all_features, get_default_feature_cols
from src.features.labeling import build_weak_labels, print_label_diagnostics
from src.utils.notebook_helpers import save_plot, save_dataframe, display_sample, setup_notebook_output
from config.settings import settings

# Configure notebook display
setup_notebook_output()

print(f"[OK] Project root: {project_root}")
print(f"[OK] Imports successful")

[OK] Notebook output configured
[OK] Project root: /Users/vinayksharma/AirDnd/cti_recommender
[OK] Imports successful


## 2. Load Raw CVE Data

In [28]:
# Load CVEs with enrichments from database
db = CVEDatabase()

query = """
SELECT 
    c.cve_id,
    c.published,
    c.modified,
    c.cvss,
    c.cvss_vector,
    c.cwe,
    e.kev_flag,
    e.epss_score,
    e.epss_percentile,
    e.is_healthcare,
    e.healthcare_score,
    e.attack_flag,
    e.attack_technique_count,
    e.chpl_flag,
    e.is_curated,
    e.curated_severity
FROM cves c
LEFT JOIN enrichments e ON c.cve_id = e.cve_id
WHERE c.cvss IS NOT NULL
ORDER BY c.published DESC
"""

df = pd.read_sql(query, db.conn)
df['published'] = pd.to_datetime(df['published'])
df['modified'] = pd.to_datetime(df['modified'])

print(f"\n{'='*70}")
print("RAW DATA LOADED")
print(f"{'='*70}")
print(f"Total CVEs: {len(df):,}")
print(f"Date range: {df['published'].min().date()} to {df['published'].max().date()}")
print(f"{'='*70}\n")

display_sample(df, n=10, title="Sample Raw Data")

2026-02-26 23:01:48 - src.core.cve_database - INFO - Connected to database
2026-02-26 23:01:48 - src.core.cve_database - INFO - Database schema created/verified

RAW DATA LOADED
Total CVEs: 210,147
Date range: 2018-01-01 to 2025-12-31



Showing 10 of 210,147 rows


,cve_id,published,modified,cvss,cvss_vector,cwe,kev_flag,...,is_healthcare,healthcare_score,attack_flag,attack_technique_count,chpl_flag,is_curated,curated_severity
0,CVE-2025-67711,2025-12-31 23:15:42.413,2026-01-06 19:03:34.700,6.1,CVSS:3.1/AV:N/AC:L/PR:N/UI:R/S:C/C:L/I:L/A:N,CWE-79,0,...,0,0.0,1,3,0,0,None
1,CVE-2025-67710,2025-12-31 23:15:42.270,2026-01-06 19:04:06.150,6.1,CVSS:3.1/AV:N/AC:L/PR:N/UI:R/S:C/C:L/I:L/A:N,CWE-79,0,...,0,0.0,1,3,0,0,None
2,CVE-2025-67709,2025-12-31 23:15:42.130,2026-01-06 19:04:27.810,6.1,CVSS:3.1/AV:N/AC:L/PR:N/UI:R/S:C/C:L/I:L/A:N,CWE-79,0,...,0,0.0,1,3,0,0,None
3,CVE-2025-67708,2025-12-31 23:15:41.980,2026-01-06 19:04:52.547,6.1,CVSS:3.1/AV:N/AC:L/PR:N/UI:R/S:C/C:L/I:L/A:N,CWE-79,0,...,0,0.0,1,3,0,0,None
4,CVE-2025-67707,2025-12-31 23:15:41.833,2026-01-06 19:08:02.547,5.6,CVSS:3.1/AV:N/AC:H/PR:N/UI:N/S:U/C:L/I:L/A:L,CWE-434,0,...,0,0.0,1,2,0,0,None
5,CVE-2025-67706,2025-12-31 23:15:41.687,2026-01-06 19:08:47.110,5.6,CVSS:3.1/AV:N/AC:H/PR:N/UI:N/S:U/C:L/I:L/A:L,CWE-434,0,...,0,0.0,1,2,0,0,None
6,CVE-2025-67705,2025-12-31 23:15:41.540,2026-01-06 19:09:08.807,6.1,CVSS:3.1/AV:N/AC:L/PR:N/UI:R/S:C/C:L/I:L/A:N,CWE-79,0,...,0,0.0,1,3,0,0,None
7,CVE-2025-67704,2025-12-31 23:15:41.387,2026-01-06 19:14:39.267,6.1,CVSS:3.1/AV:N/AC:L/PR:N/UI:R/S:C/C:L/I:L/A:N,CWE-79,0,...,0,0.0,1,3,0,0,None
8,CVE-2025-67703,2025-12-31 23:15:40.540,2026-01-06 19:15:11.537,6.1,CVSS:3.1/AV:N/AC:L/PR:N/UI:R/S:C/C:L/I:L/A:N,CWE-79,0,...,0,0.0,1,3,0,0,None
9,CVE-2025-69288,2025-12-31 22:15:49.410,2026-01-13 15:25:44.200,9.1,CVSS:3.1/AV:N/AC:L/PR:H/UI:N/S:C/C:H/I:H/A:H,CWE-20,0,...,0,0.0,1,2,0,0,None


... 210,137 more rows


## 3. Feature Engineering

Create ML features from raw CVE data using `src.features.engineering`

In [29]:
# Create all features using modular function
print("Creating features...")
feature_cols = get_default_feature_cols()
df_features = create_all_features(df, feature_cols)

print(f"\n{'='*70}")
print("FEATURE ENGINEERING COMPLETE")
print(f"{'='*70}")
print(f"Input rows: {len(df):,}")
print(f"Output rows: {len(df_features):,}")
print(f"Features used: {len(feature_cols)}")
print(f"{'='*70}\n")

# Show feature names
print(f"[STATS] Engineered Features ({len(feature_cols)}):")
for i, col in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {col}")

display_sample(df_features[['cve_id'] + feature_cols[:10]], n=10, title="Sample Features")

Creating features...
Feature engineering complete: 210,147 rows
Columns: 16 original -> 27 total (+11 new features)

[OK] 11 NEW features created (0% missing):
   - cvss_epss_product
   - cvss_missing_flag
   - cvss_norm
   - days_since_published
   - epss_missing_flag
   - epss_percentile_missing_flag
   - has_attack
   - kev_healthcare_interaction
   - published_missing
   - published_week
   - recency_score

[STATS] No missingness in original columns (data already clean)
Feature engineering complete: 210,147 rows, 16 features

Feature statistics:
                                   mean       std   min        max
cvss_norm                        0.6879    0.1716   0.0     1.0000
epss_score                       0.0000    0.0000   0.0     0.0000
epss_percentile                  0.4347    0.2765   0.0     1.0000
kev_flag                         0.0056    0.0746   0.0     1.0000
days_since_published          1218.6734  833.0328  56.0  2978.0000
recency_score                    0.5908   

Showing 10 of 210,147 rows


,cve_id,cvss_norm,epss_score,epss_percentile,kev_flag,days_since_published,recency_score,attack_technique_count,has_attack,chpl_flag,is_healthcare
0,CVE-2025-67711,0.61,0.0,0.13274,0,56,0.981195,3,1,0,0
1,CVE-2025-67710,0.61,0.0,0.13274,0,56,0.981195,3,1,0,0
2,CVE-2025-67709,0.61,0.0,0.13274,0,56,0.981195,3,1,0,0
3,CVE-2025-67708,0.61,0.0,0.13274,0,56,0.981195,3,1,0,0
4,CVE-2025-67707,0.56,0.0,0.35216,0,56,0.981195,2,1,0,0
5,CVE-2025-67706,0.56,0.0,0.35216,0,56,0.981195,2,1,0,0
6,CVE-2025-67705,0.61,0.0,0.13274,0,56,0.981195,3,1,0,0
7,CVE-2025-67704,0.61,0.0,0.13274,0,56,0.981195,3,1,0,0
8,CVE-2025-67703,0.61,0.0,0.13274,0,56,0.981195,3,1,0,0
9,CVE-2025-69288,0.91,0.0,0.66214,0,56,0.981195,2,1,0,0


... 210,137 more rows


## 4. Feature Missingness Analysis

In [30]:
# Analyze missing values in features
print(f"\n{'='*70}")
print("FEATURE MISSINGNESS REPORT")
print(f"{'='*70}")

# Calculate completeness for each feature
missing_report = {}
for col in feature_cols:
    total = len(df_features)
    missing = df_features[col].isna().sum()
    pct_complete = ((total - missing) / total) * 100
    missing_report[col] = {
        'missing': missing,
        'complete': total - missing,
        'pct_complete': pct_complete
    }

# Sort by completeness
sorted_features = sorted(missing_report.items(), key=lambda x: x[1]['pct_complete'])
for feat, stats in sorted_features:
    status = "[OK]" if stats['pct_complete'] >= 95 else "[WARN]" if stats['pct_complete'] >= 70 else "[FAIL]"
    print(f"  {status} {feat:30s}: {stats['pct_complete']:5.1f}% complete")


FEATURE MISSINGNESS REPORT
  [OK] cvss_norm                     : 100.0% complete
  [OK] epss_score                    : 100.0% complete
  [OK] epss_percentile               : 100.0% complete
  [OK] kev_flag                      : 100.0% complete
  [OK] days_since_published          : 100.0% complete
  [OK] recency_score                 : 100.0% complete
  [OK] attack_technique_count        : 100.0% complete
  [OK] has_attack                    : 100.0% complete
  [OK] chpl_flag                     : 100.0% complete
  [OK] is_healthcare                 : 100.0% complete
  [OK] cvss_epss_product             : 100.0% complete
  [OK] kev_healthcare_interaction    : 100.0% complete
  [OK] published_missing             : 100.0% complete
  [OK] cvss_missing_flag             : 100.0% complete
  [OK] epss_missing_flag             : 100.0% complete
  [OK] epss_percentile_missing_flag  : 100.0% complete


In [31]:
# Visualize feature completeness
completeness = {}
for col in feature_cols:
    completeness[col] = (df_features[col].notna().sum() / len(df_features)) * 100

completeness_df = pd.DataFrame({
    'Feature': list(completeness.keys()),
    'Completeness': list(completeness.values())
}).sort_values('Completeness')

fig = px.bar(
    completeness_df,
    x='Completeness',
    y='Feature',
    orientation='h',
    title='Feature Completeness (%)',
    labels={'Completeness': 'Completeness (%)', 'Feature': ''},
    color='Completeness',
    color_continuous_scale='RdYlGn'
)
fig.update_layout(height=max(400, len(feature_cols) * 25))

save_plot(fig, 'feature_completeness')

print(f"\n[OK] Feature completeness plot saved")


[OK] Feature completeness plot saved


## 5. Weak Label Construction

Build graded labels (0-5) with confidence scores based on multiple risk signals

In [32]:
# Construct weak labels using modular function
print("Constructing weak labels...")
df_labeled = build_weak_labels(df_features)

print(f"\n{'='*70}")
print("WEAK LABEL CONSTRUCTION COMPLETE")
print(f"{'='*70}")
print(f"Total CVEs: {len(df_labeled):,}")
print(f"Labeled CVEs: {df_labeled['soft_label'].notna().sum():,}")
print(f"{'='*70}\n")

# Label distribution
label_dist = df_labeled['soft_label'].value_counts().sort_index()
print(f"[STATS] Label Distribution:")
for label, count in label_dist.items():
    pct = (count / len(df_labeled)) * 100
    print(f"  Label {label}: {count:7,} ({pct:5.2f}%)")

# Confidence stats
print(f"\n Confidence Statistics:")
print(f"  Mean: {df_labeled['label_confidence'].mean():.3f}")
print(f"  Median: {df_labeled['label_confidence'].median():.3f}")
print(f"  High confidence (≥0.7): {(df_labeled['label_confidence'] >= 0.7).sum():,} ({(df_labeled['label_confidence'] >= 0.7).mean()*100:.1f}%)")

Constructing weak labels...

WEAK LABEL CONSTRUCTION COMPLETE
Total CVEs: 210,147
Labeled CVEs: 210,147

[STATS] Label Distribution:
  Label 0:  70,459 (33.53%)
  Label 1: 133,937 (63.73%)
  Label 2:   5,707 ( 2.72%)
  Label 3:      44 ( 0.02%)

 Confidence Statistics:
  Mean: 0.323
  Median: 0.300
  High confidence (≥0.7): 10,819 (5.1%)


## 6. Label Diagnostics

In [33]:
# Print comprehensive label diagnostics
print_label_diagnostics(df_labeled)

LABEL DISTRIBUTION DIAGNOSTICS

1. SOFT LABEL DISTRIBUTION
----------------------------------------
  Label 0:   70,459 (33.53%) #################################
  Label 1:  133,937 (63.73%) ###############################################################
  Label 2:    5,707 ( 2.72%) ##
  Label 3:       44 ( 0.02%) 

2. LABEL SOURCE BREAKDOWN
----------------------------------------
  default             :   70,459 (33.53%)
  attack_mapping      :   64,921 (30.89%)
  cvss_recency        :   38,645 (18.39%)
  medium_epss         :   30,371 (14.45%)
  epss_attack         :    4,574 ( 2.18%)
  kev_only            :    1,133 ( 0.54%)
  kev_healthcare      :       44 ( 0.02%)

3. LABEL CONFIDENCE STATISTICS
----------------------------------------
  Min:    0.200
  Mean:   0.323
  Median: 0.300
  Max:    1.000
  Std:    0.176

  Confidence Distribution:
    0.0-0.2:        0 ( 0.00%) 
    0.2-0.4:  164,127 (78.10%) #######################################
    0.4-0.6:   23,767 (11.31%) #####

In [34]:
# Visualize label distribution
label_counts = df_labeled['soft_label'].value_counts().sort_index()

fig = px.bar(
    x=label_counts.index.astype(str),
    y=label_counts.values,
    title='Weak Label Distribution',
    labels={'x': 'Label (0=Low Risk, 3=Critical)', 'y': 'Number of CVEs'},
    text=label_counts.values,
    color=label_counts.values,
    color_continuous_scale='YlOrRd'
)
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(height=450, showlegend=False)

save_plot(fig, 'weak_label_distribution')

print("\n[OK] Label distribution plot saved")


[OK] Label distribution plot saved


In [35]:
# Analyze confidence distribution
fig = px.histogram(
    df_labeled,
    x='label_confidence',
    nbins=50,
    title='Label Confidence Distribution',
    labels={'label_confidence': 'Confidence Score', 'count': 'Number of CVEs'},
    color_discrete_sequence=['#3498DB']
)
fig.update_layout(height=400)

save_plot(fig, 'label_confidence_distribution')

print(f"\n[STATS] Confidence Statistics:")
print(f"  Mean: {df_labeled['label_confidence'].mean():.3f}")
print(f"  Median: {df_labeled['label_confidence'].median():.3f}")
print(f"  High confidence (>0.8): {(df_labeled['label_confidence'] > 0.8).sum():,} ({(df_labeled['label_confidence'] > 0.8).mean()*100:.1f}%)")
print(f"  Low confidence (<0.5): {(df_labeled['label_confidence'] < 0.5).sum():,} ({(df_labeled['label_confidence'] < 0.5).mean()*100:.1f}%)")


[STATS] Confidence Statistics:
  Mean: 0.323
  Median: 0.300
  High confidence (>0.8): 5,843 (2.8%)
  Low confidence (<0.5): 169,622 (80.7%)


## 7. Feature Correlations with Labels

In [36]:
# Calculate correlations between features and labels
numeric_features = df_labeled[feature_cols].select_dtypes(include=[np.number]).columns.tolist()

if 'soft_label' in df_labeled.columns:
    # Calculate correlations
    correlations = df_labeled[numeric_features + ['soft_label']].corr()['soft_label'].drop('soft_label')
    
    # Remove NaN correlations (features with no variance)
    valid_correlations = correlations.dropna().sort_values(ascending=False)
    
    # Identify problematic features
    nan_features = correlations[correlations.isna()].index.tolist()
    
    print(f"\n{'='*70}")
    print("FEATURE-LABEL CORRELATIONS")
    print(f"{'='*70}")
    
    if nan_features:
        print(f"\n[WARN] Features with no variance (excluded from correlation):")
        for feat in nan_features:
            unique_vals = df_labeled[feat].nunique()
            print(f"  • {feat}: {unique_vals} unique value(s)")
        print()
    
    print(f"Top 10 positively correlated features:")
    for feat, corr in valid_correlations.head(10).items():
        print(f"  {feat:30s}: {corr:+.3f}")
    
    if len(valid_correlations) > 10:
        print(f"\nTop 5 negatively correlated features:")
        for feat, corr in valid_correlations.tail(5).items():
            print(f"  {feat:30s}: {corr:+.3f}")
    print(f"{'='*70}\n")
    
    # Visualize top correlations (only valid ones)
    top_corrs = pd.concat([valid_correlations.head(10), valid_correlations.tail(5)]).sort_values()
    
    fig = px.bar(
        x=top_corrs.values,
        y=top_corrs.index,
        orientation='h',
        title='Feature Correlations with Label',
        labels={'x': 'Correlation', 'y': 'Feature'},
        color=top_corrs.values,
        color_continuous_scale='RdBu_r'
    )
    fig.update_layout(height=500, showlegend=False)
    
    save_plot(fig, 'feature_label_correlations')
    print("[OK] Feature-label correlation plot saved")


FEATURE-LABEL CORRELATIONS

[WARN] Features with no variance (excluded from correlation):
  • epss_score: 1 unique value(s)
  • cvss_epss_product: 1 unique value(s)
  • published_missing: 1 unique value(s)
  • cvss_missing_flag: 1 unique value(s)
  • epss_missing_flag: 1 unique value(s)
  • epss_percentile_missing_flag: 1 unique value(s)

Top 10 positively correlated features:
  has_attack                    : +0.567
  attack_technique_count        : +0.480
  cvss_norm                     : +0.344
  epss_percentile               : +0.285
  kev_flag                      : +0.195
  recency_score                 : +0.122
  chpl_flag                     : +0.031
  kev_healthcare_interaction    : +0.014
  is_healthcare                 : +0.001
  days_since_published          : -0.122

[OK] Feature-label correlation plot saved


## 8. Label Quality Analysis

In [37]:
# Analyze label quality by checking signal combinations
print(f"\n{'='*70}")
print("LABEL QUALITY ANALYSIS")
print(f"{'='*70}")

# High confidence, high priority labels
high_priority = df_labeled[(df_labeled['soft_label'] >= 2) & (df_labeled['label_confidence'] > 0.8)]
print(f"\n High Priority CVEs (label≥2, confidence>0.8):")
print(f"   Count: {len(high_priority):,}")
print(f"   KEV: {high_priority['kev_flag'].sum():,} ({high_priority['kev_flag'].mean()*100:.1f}%)")
print(f"   Healthcare: {high_priority['is_healthcare'].sum():,} ({high_priority['is_healthcare'].mean()*100:.1f}%)")
print(f"   Mean CVSS: {high_priority['cvss'].mean():.2f}")

# Low confidence labels (need manual review)
low_confidence = df_labeled[df_labeled['label_confidence'] < 0.4]
print(f"\n[WARN]  Low Confidence CVEs (confidence<0.4):")
print(f"   Count: {len(low_confidence):,}")
print(f"   Label distribution: {dict(low_confidence['soft_label'].value_counts().sort_index())}")

# Multi-signal CVEs (most reliable)
multi_signal = df_labeled[
    (df_labeled['kev_flag'] == 1) | 
    (df_labeled['is_healthcare'] == 1) | 
    (df_labeled['attack_flag'] == 1)
]
print(f"\n[OK] Multi-Signal CVEs (KEV or Healthcare or ATT&CK):")
print(f"   Count: {len(multi_signal):,}")
print(f"   Mean label: {multi_signal['soft_label'].mean():.2f}")
print(f"   Mean confidence: {multi_signal['label_confidence'].mean():.3f}")

print(f"{'='*70}\n")


LABEL QUALITY ANALYSIS

 High Priority CVEs (label≥2, confidence>0.8):
   Count: 5,751
   KEV: 1,177 (20.5%)
   Healthcare: 16 (0.3%)
   Mean CVSS: 7.98

[WARN]  Low Confidence CVEs (confidence<0.4):
   Count: 164,127
   Label distribution: {0: np.int64(66056), 1: np.int64(98071)}

[OK] Multi-Signal CVEs (KEV or Healthcare or ATT&CK):
   Count: 82,711
   Mean label: 1.07
   Mean confidence: 0.394



## 9. Save Processed Features

In [38]:
# Save processed features with labels for model training
output_path = save_dataframe(
    df_labeled,
    name=f'features_with_labels_{pd.Timestamp.now().strftime("%Y%m%d")}',
    subdir='features',
    format='csv'  # CSV format for compatibility
)

print(f"\n[OK] Processed features saved: {output_path}")
print(f"  Rows: {len(df_labeled):,}")
print(f"  Columns: {len(df_labeled.columns)}")
print(f"  File size: {output_path.stat().st_size / (1024**2):.2f} MB")

[OK] DataFrame saved: outputs/features/features_with_labels_20260226.csv

[OK] Processed features saved: outputs/features/features_with_labels_20260226.csv
  Rows: 210,147
  Columns: 30
  File size: 47.57 MB


## 10. Feature Summary Statistics

In [39]:
# Generate summary statistics for all features
print(f"\n{'='*70}")
print("FEATURE ENGINEERING SUMMARY")
print(f"{'='*70}")

print(f"\n[STATS] Dataset:")
print(f"  Total CVEs: {len(df_labeled):,}")
print(f"  Features created: {len(feature_cols)}")
print(f"  Date range: {df_labeled['published'].min().date()} to {df_labeled['published'].max().date()}")

print(f"\n  Weak Labels:")
print(f"  Labels: 0 (Low) to 3 (Critical)")
print(f"  Mean label: {df_labeled['soft_label'].mean():.2f}")
print(f"  Mean confidence: {df_labeled['label_confidence'].mean():.3f}")
print(f"  High priority (label≥2): {(df_labeled['soft_label'] >= 2).sum():,} ({(df_labeled['soft_label'] >= 2).mean()*100:.1f}%)")

print(f"\n Feature Completeness:")
avg_completeness = completeness_df['Completeness'].mean()
print(f"  Average: {avg_completeness:.1f}%")
print(f"  Complete features (>90%): {(completeness_df['Completeness'] > 90).sum()}")
print(f"  Partial features (50-90%): {((completeness_df['Completeness'] >= 50) & (completeness_df['Completeness'] <= 90)).sum()}")

print(f"\n Outputs:")
print(f"  Features: outputs/features/features_with_labels_*.csv")
print(f"  Plots: outputs/plots/")

print(f"\n{'='*70}")

# Close database
db.conn.close()
print("\n[OK] Feature Engineering Complete")


FEATURE ENGINEERING SUMMARY

[STATS] Dataset:
  Total CVEs: 210,147
  Features created: 16
  Date range: 2018-01-01 to 2025-12-31

  Weak Labels:
  Labels: 0 (Low) to 3 (Critical)
  Mean label: 0.69
  Mean confidence: 0.323
  High priority (label≥2): 5,751 (2.7%)

 Feature Completeness:
  Average: 100.0%
  Complete features (>90%): 16
  Partial features (50-90%): 0

 Outputs:
  Features: outputs/features/features_with_labels_*.csv
  Plots: outputs/plots/


[OK] Feature Engineering Complete


## Next Steps

1. **Model Training** -> Run `Model_Training_And_Evaluation.ipynb` to:
   - Create temporal train/val/test splits
   - Train confidence-weighted LambdaMART
   - Compare with baseline and advanced models
   - Evaluate performance and explainability

---